# Practical 5 — Cost Functions for Regression and Regularisation

**Lectures:** 9. Cost Functions for Regression · 11. Regularisation

Part A looks at **what we minimise**: MSE, MAE, Huber, log-cosh and quantile losses, and how the choice
changes a model's behaviour when the data contain **outliers**, or when we want a **prediction interval**
rather than a single point prediction.

Part B adds a **penalty on the size of the coefficients** — **Ridge (L2)** and **LASSO (L1)**
regularisation — to control overfitting and to perform feature selection.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)

---
# Part A — Cost functions
## A1. The losses as functions of the residual (≈15 min)
For a residual $r = \hat y - y$, the per-example losses are:

| name | $L(r)$ |
|---|---|
| squared (MSE) | $r^2$ |
| absolute (MAE) | $\lvert r\rvert$ |
| Huber, threshold $\delta$ | $\tfrac12 r^2$ if $\lvert r\rvert\le\delta$, else $\delta(\lvert r\rvert - \tfrac12\delta)$ |
| log-cosh | $\log\cosh(r)$ |
| quantile (pinball), level $q$, with $e = y - \hat y$ | $\max(q\,e,\;(q-1)\,e)$ |

**TODO A1.1** Implement each function (vectorised, taking an array of residuals) and plot them all for
$r\in[-4,4]$ on one set of axes (use $\delta = 1$ and $q=0.9$). Which losses grow **quadratically**, which
**linearly** for large residuals? Which is not symmetric?

In [ ]:
def squared(r):
    # YOUR CODE HERE
    raise NotImplementedError

def absolute(r):
    # YOUR CODE HERE
    raise NotImplementedError

def huber(r, delta=1.0):
    # YOUR CODE HERE
    raise NotImplementedError

def log_cosh(r):
    # YOUR CODE HERE
    raise NotImplementedError

def quantile_loss(r, q=0.9):
    """r = y_hat - y, so the error e = y - y_hat = -r."""
    # YOUR CODE HERE
    raise NotImplementedError

r = np.linspace(-4, 4, 400)
# YOUR CODE HERE

## A2. Outliers: MSE vs MAE vs Huber (≈25 min)
We generate a clean linear relationship and then corrupt 10 % of the targets with large outliers.

**TODO A2.1** Write a gradient-descent fitter `fit_linear(x, y, loss_grad)` for a line
$\hat y = \theta_0 + \theta_1 x$, where `loss_grad(r)` returns $\partial L/\partial r$ for each
residual. Using the chain rule, $\partial J/\partial\theta_0 = \frac1m\sum L'(r_i)$ and
$\partial J/\partial\theta_1 = \frac1m\sum L'(r_i)x_i$. Supply the derivatives:

* squared: $2r$ · absolute: $\operatorname{sign}(r)$ · Huber: $r$ if $|r|\le\delta$ else $\delta\operatorname{sign}(r)$ · log-cosh: $\tanh(r)$

In [ ]:
m = 100
x = rng.uniform(0, 10, m)
y = 2 + 0.5 * x + rng.normal(0, 0.5, m)
out = rng.choice(m, size=10, replace=False)
y_out = y.copy(); y_out[out] += rng.uniform(8, 15, 10)          # outliers push the target upwards

def grad_mse(r):
    # YOUR CODE HERE
    raise NotImplementedError

def grad_mae(r):
    # YOUR CODE HERE
    raise NotImplementedError

def grad_huber(r, delta=1.0):
    # YOUR CODE HERE
    raise NotImplementedError

def grad_log_cosh(r):
    # YOUR CODE HERE
    raise NotImplementedError

grads = {"MSE": grad_mse, "MAE": grad_mae, "Huber": grad_huber, "log-cosh": grad_log_cosh}

def fit_linear(x, y, loss_grad, alpha=0.01, n_iters=20000):
    t0 = t1 = 0.0
    for _ in range(n_iters):
        # YOUR CODE HERE
        raise NotImplementedError
    return t0, t1

fits = {name: fit_linear(x, y_out, g) for name, g in grads.items()}
print(pd.DataFrame(fits, index=["theta0", "theta1"]).T.round(3), "\n(truth: theta0=2, theta1=0.5)")

**TODO A2.2** Plot the data (highlight the outliers) and the four fitted lines. Then check your MSE and
Huber fits against `LinearRegression` and `HuberRegressor(epsilon=1.35)`. Which loss would you choose
here and why? When might you *want* the model to be influenced by extreme values?

In [ ]:
from sklearn.linear_model import LinearRegression, HuberRegressor

# YOUR CODE HERE

## A3. Quantile regression and prediction intervals (≈15 min)
Minimising the quantile loss at level $q$ gives a model of the $q$-th **conditional quantile** of $y$.
Fitting $q = 0.05$ and $q=0.95$ gives a 90 % **prediction interval** — useful when the noise is not
constant (*heteroscedastic*).

**TODO A3.1** Using `QuantileRegressor(quantile=q, alpha=0, solver="highs")`, fit $q \in\{0.05, 0.5, 0.95\}$
to the data below and plot the three lines. What fraction of points lies inside your interval? How does
this differ from a *confidence interval* for the mean?

In [ ]:
from sklearn.linear_model import QuantileRegressor

xh = rng.uniform(0, 10, 400)
yh = 1 + 0.8 * xh + rng.normal(0, 0.2 + 0.3 * xh)          # noise grows with x
# YOUR CODE HERE

---
# Part B — Regularisation
$$J(\boldsymbol\theta) = \frac{1}{2m}\lVert X\boldsymbol\theta - \mathbf y\rVert^2 + \lambda\,\Omega(\boldsymbol\theta),\qquad
\Omega_{\text{ridge}} = \sum_{j\ge1}\theta_j^2,\quad \Omega_{\text{lasso}} = \sum_{j\ge1}|\theta_j|$$
(the bias $\theta_0$ is not penalised). Ridge has a closed form:
$\boldsymbol\theta = (X^\top X + \lambda' I_0)^{-1}X^\top\mathbf y$, where $I_0$ is the identity with a 0 in the bias position.

## B1. Ridge from scratch to tame an overfitting polynomial (≈20 min)
**TODO B1.1** Implement `ridge_fit(Xb, y, lam)` using the closed form (with `np.linalg.solve`).
Fit a degree-15 polynomial to 30 noisy samples of $\sin(2\pi x)$ (standardise the polynomial features)
with $\lambda' \in \{0, 10^{-6}, 10^{-3}, 1\}$, and plot the four curves. Compute the validation RMSE of each.

In [ ]:
def ridge_fit(Xb, y, lam):
    """Xb includes a leading column of ones; the bias is not penalised."""
    # YOUR CODE HERE
    raise NotImplementedError

def make_sine(n, seed):
    g = np.random.default_rng(seed); x = g.uniform(0, 1, n)
    return x, np.sin(2 * np.pi * x) + g.normal(0, 0.25, n)

x_tr, y_tr = make_sine(30, 1); x_va, y_va = make_sine(1000, 2)
P = lambda x: np.column_stack([x ** p for p in range(1, 16)])
sc = StandardScaler().fit(P(x_tr))
design = lambda x: np.column_stack([np.ones(len(x)), sc.transform(P(x))])

xs = np.linspace(0, 1, 300)
plt.scatter(x_tr, y_tr, c="k", s=15)
for lam in [0, 1e-6, 1e-3, 1]:
    # YOUR CODE HERE
    raise NotImplementedError
plt.ylim(-2, 2); plt.legend(fontsize=8); plt.show()

## B2. Linear vs Ridge vs LASSO on real data (≈25 min)
The **white wine quality** dataset has 11 physico-chemical features and a quality score (0–10).

**TODO B2.1** Split 80/20 (`random_state=0`), standardise, and fit `LinearRegression`, `Ridge(alpha=1)` and
`Lasso(alpha=0.02)`. Report the test RMSE and $R^2$ of each and show all three coefficient vectors
side by side. Which coefficients does LASSO set **exactly to zero**?

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

wine = pd.read_csv("https://raw.githubusercontent.com/lshi/ml/main/data/winequality-white.csv", sep=";")
Xw, yw = wine.drop(columns="quality"), wine["quality"].to_numpy()
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.2, random_state=0)
scw = StandardScaler().fit(Xw_tr)
Xw_tr_s, Xw_te_s = scw.transform(Xw_tr), scw.transform(Xw_te)

# YOUR CODE HERE

**TODO B2.2 — Regularisation paths.** For $\alpha$ on a log grid from $10^{-4}$ to $10^{1}$ (Lasso)
and $10^{-2}$ to $10^{5}$ (Ridge), record the coefficients and plot each coefficient against $\alpha$
(log x-axis), one subplot per method. Describe the difference in how the coefficients shrink.
Which features survive longest under LASSO?

In [ ]:
# YOUR CODE HERE

## B3. Extension
* Implement LASSO yourself with **coordinate descent** using the soft-thresholding operator
  $S(z,\gamma) = \operatorname{sign}(z)\max(|z|-\gamma, 0)$, and check it against `Lasso` (note scikit-learn's cost is $\frac{1}{2m}\lVert\cdot\rVert^2 + \alpha\lVert\theta\rVert_1$).
* Choose $\alpha$ for Ridge and LASSO with `RidgeCV` / `LassoCV` (cross-validation — see Practical 6).

In [ ]:
# YOUR CODE HERE

## Lab report checklist (Practical 5)
1. The loss-function plot, and one sentence per loss on when you would use it.
2. The outlier experiment: a figure of the fitted lines and a table of fitted parameters; which loss is most robust and why (relate to the gradient of each loss).
3. The quantile-regression figure and the coverage you measured; the difference between a prediction interval and a confidence interval.
4. Test RMSE/$R^2$ and coefficients for Linear, Ridge and LASSO on the wine data; the regularisation-path figure; which features LASSO selects.